In [1]:
!git clone https://github.com/Abdelwhabmohammed/akt26_gpu_dev

Cloning into 'akt26_gpu_dev'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 65 (delta 4), reused 7 (delta 1), pack-reused 51 (from 1)
Receiving objects: 100% (65/65), 18.38 KiB | 4.59 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [2]:
%cd /content/akt26_gpu_dev/3-ger

/content/akt26_gpu_dev/3-ger


In [3]:
!nvcc -gencode arch=compute_75,code=sm_75 \
  -gencode arch=compute_75,code=compute_75 \
  ger.cu -o ger -L/usr/local/cuda/lib64 -lcublas && ./ger


Matrix: 4000 x 2000, alpha = 1.50, repeats = 100
Matrix storage: column-major

Implementation          Time (us)      GFLOP/s           Error Norm
--------------------------------------------------------------------
Custom kernel             318.709       50.203         4.440474e-08
cuBLAS SGER               324.992       49.232         4.440474e-08

TEST PASS
Speedup (cuBLAS time / custom time): 1.020x


In [4]:
%cd ..

/content/akt26_gpu_dev


In [5]:
%cd /content/akt26_gpu_dev/4-gemm

/content/akt26_gpu_dev/4-gemm


In [6]:
!nvcc -gencode arch=compute_75,code=sm_75 \
  -gencode arch=compute_75,code=compute_75 \
  gemm.cu -o gemm -L/usr/local/cuda/lib64 -lcublas && ./gemm

GPU: Tesla T4 (compute capability 7.5)
Matrix size: 16 x 16, batch size: 1000
FP32, column-major matrices, repeats: 100

Implementation              Time (us)      GFLOP/s  Max batch fwd error
-----------------------------------------------------------------------
Custom kernel                  22.392      365.840         2.438560e-08
cuBLAS Strided Batched         48.885      167.578         3.199161e-08

TEST PASS
Speedup (cuBLAS time / custom time): 2.183x


In [7]:
!nvcc -gencode arch=compute_75,code=sm_75 \
  -gencode arch=compute_75,code=compute_75 \
  gemm_shmem.cu -o gemm_shmem -L/usr/local/cuda/lib64 -lcublas && ./gemm_shmem

GPU: Tesla T4 (compute capability 7.5)
Matrix size: 16 x 16, batch size: 1000
FP32, column-major matrices, repeats: 100

Implementation              Time (us)      GFLOP/s  Max batch fwd error
-----------------------------------------------------------------------
Custom kernel                  30.287      270.482         2.438560e-08
cuBLAS Strided Batched         61.357      133.513         3.199161e-08

TEST PASS
Speedup (cuBLAS time / custom time): 2.026x


## CUDA Kernels — GER &amp; Batched GEMM on GPU

---

### Environment

| Item | Value |
|---|---|
| **GPU** | Tesla T4 (compute capability 7.5, Turing) |
| **Platform** | Google Colab |
| **Compiler** | `nvcc` with `-gencode arch=compute_75,code=sm_75` |
| **CUDA libraries** | cuBLAS (strided batched GEMM, SGER) |

---

### Results — Task 3: Rank-1 Update (GER)

**Kernel:** 2D grid of 16×16 thread blocks; each thread computes one element `A[i,j] += α·x[i]·y[j]` (column-major).

| | Custom Kernel | cuBLAS `SGER` |
|---|---|---|
| **Matrix** | 4000 × 2000, α = 1.50, 100 repeats | same |
| **Time (μs)** | 318.7 | 325.0 |
| **GFLOP/s** | 50.2 | 49.2 |
| **Error norm** | 4.44e-08 | 4.44e-08 |
| **Speedup** | **1.020×** vs cuBLAS | — |

---

### Results — Task 4: Batched GEMM (Simple — Global Memory)

**Kernel:** One block per batch element, N×N threads per block; each thread computes one element of C via an inner dot-product loop reading directly from global memory.

| | Custom Kernel | cuBLAS Strided Batched |
|---|---|---|
| **Matrix** | 16 × 16, batch = 1000, 100 repeats | same |
| **Time (μs)** | 22.4 | 48.9 |
| **GFLOP/s** | 365.8 | 167.6 |
| **Max batch fwd error** | 2.44e-08 | 3.20e-08 |
| **Speedup** | **2.183×** vs cuBLAS | — |

---

### Results — Task 4: Batched GEMM (Shared Memory)

**Kernel:** Same structure, but A[b] and B[b] are loaded into `__shared__` memory with `__syncthreads()` before the dot-product loop.

| | Custom Kernel | cuBLAS Strided Batched |
|---|---|---|
| **Matrix** | 16 × 16, batch = 1000, 100 repeats | same |
| **Time (μs)** | 30.3 | 61.4 |
| **GFLOP/s** | 270.5 | 133.5 |
| **Max batch fwd error** | 2.44e-08 | 3.20e-08 |
| **Speedup** | **2.026×** vs cuBLAS | — |

---

## Summary & Observations

| Metric | GER | GEMM (simple) | GEMM (shared mem) |
|---|---|---|---|
| Speedup vs cuBLAS | 1.020× | **2.183×** | 2.026× |
| GFLOP/s | 50.2 | **365.8** | 270.5 |

**Key conclusions:**
- **GER** matches cuBLAS — the rank-1 update is memory-bandwidth-bound; the custom 2D-grid kernel achieves near-identical throughput.
- **Batched GEMM (simple)** outperforms cuBLAS by 2.18× — for tiny N ≤ 32 matrices, launch overhead dominates cuBLAS; the one-block-per-batch custom kernel avoids it.
- **Shared memory did not help** for N = 16 — the entire 16×16 matrix (1 KB) fits in L1/registers, so explicitly staging through `__shared__` adds `__syncthreads()` overhead without reducing memory traffic. For larger N (e.g., 32), shared memory would show benefit.
- Both custom GEMM kernels beat cuBLAS because the problem is too small for cuBLAS's internal heuristics and scheduling to amortize.

